# Extract mentions

Joins `risk/risks-v2.tsv` to `risk/studies.tsv` and writes two derived CSVs. The source TSVs are not changed.

- `risk/mentions.csv` — one row per risk coding: `id`, `label`, `ev_id`, `publication_id`
- `risk/ev_titles.csv` — one row per evidence id: `ev_id`, `publication_id`, `title`

`Paper_ID` is the publication id. A blank `Paper_ID` is filled from the single id already used by that title. Run the cells from the top.

In [1]:
from pathlib import Path

import polars as pl

RISKS = Path("risk", "risks-v2.tsv")
STUDIES = Path("risk", "studies.tsv")
MENTIONS = Path("risk", "mentions.csv")
EV_TITLES = Path("risk", "ev_titles.csv")

pl.Config.set_tbl_rows(-1)
pl.Config.set_fmt_str_lengths(500)
pl.Config.set_tbl_width_chars(200)

def show(frame: pl.DataFrame) -> None:
    if frame.height == 0:
        print("none")
    else:
        print(frame)

In [2]:
risks = pl.read_csv(
    RISKS,
    separator="\t",
    schema_overrides={
        "RISK_ID": pl.Int64,
        "Assotiated Ev_ID": pl.String,
        "Risk": pl.String,
        "Comment": pl.String,
    },
).with_row_index("_order")

print(f"{risks.height} risk rows")

351 risk rows


In [3]:
studies = pl.read_csv(
    STUDIES,
    separator="\t",
    schema_overrides={
        "Title": pl.String,
        "QuickRef": pl.String,
        "Ev_ID": pl.String,
        "Paper_ID": pl.Int64,
        "Cat_ID": pl.Int64,
    },
).with_row_index("_order").with_columns(
    pl.col("Title").str.strip_chars().alias("title"),
    pl.col("Ev_ID").str.strip_chars().alias("ev_id"),
)

print(f"{studies.height} study rows")

146 study rows


In [4]:
studies

_order,Title,QuickRef,Ev_ID,Paper_ID,Cat_ID,SubCat_ID,AddEv_ID,title,ev_id
u32,str,str,str,i64,i64,i64,str,str,str
0,"""A framework for ethical Ai at the United Nations""","""Hogenhout2021""","""06.08.00""",6,8,null,null,"""A framework for ethical Ai at the United Nations""","""06.08.00"""
1,"""A framework for ethical Ai at the United Nations""","""Hogenhout2021""","""06.10.00""",6,10,null,null,"""A framework for ethical Ai at the United Nations""","""06.10.00"""
2,"""Examining the differential risk from high-level artificial intelligence and the question of control""","""Kilian2023""","""07.03.00""",7,3,null,null,"""Examining the differential risk from high-level artificial intelligence and the question of control""","""07.03.00"""
3,"""Managing the ethical and risk implications of rapid advances in artificial intelligence: A literature review""","""Meek2016""","""09.01.01""",9,1,1,null,"""Managing the ethical and risk implications of rapid advances in artificial intelligence: A literature review""","""09.01.01"""
4,"""Managing the ethical and risk implications of rapid advances in artificial intelligence: A literature review""","""Meek2016""","""09.02.07""",9,2,7,null,"""Managing the ethical and risk implications of rapid advances in artificial intelligence: A literature review""","""09.02.07"""
5,"""Managing the ethical and risk implications of rapid advances in artificial intelligence: A literature review""","""Meek2016""","""09.03.01""",9,3,1,null,"""Managing the ethical and risk implications of rapid advances in artificial intelligence: A literature review""","""09.03.01"""
6,"""Managing the ethical and risk implications of rapid advances in artificial intelligence: A literature review""","""Meek2016""","""09.04.02""",9,4,2,null,"""Managing the ethical and risk implications of rapid advances in artificial intelligence: A literature review""","""09.04.02"""
7,"""Managing the ethical and risk implications of rapid advances in artificial intelligence: A literature review""","""Meek2016""","""09.06.02""",9,6,2,null,"""Managing the ethical and risk implications of rapid advances in artificial intelligence: A literature review""","""09.06.02"""
8,"""Taxonomy of Risks posed by Language Models""","""Weidinger2022""","""16.05.04""",16,5,4,null,"""Taxonomy of Risks posed by Language Models""","""16.05.04"""


In [5]:
paper_for_title = (
    studies.filter(pl.col("Paper_ID").is_not_null())
    .group_by("title")
    .agg(pl.col("Paper_ID").unique().sort())
    .filter(pl.col("Paper_ID").list.len() == 1)
    .select("title", pl.col("Paper_ID").list.first().alias("filled_paper_id"))
)

studies = studies.join(paper_for_title, on="title", how="left").with_columns(
    pl.coalesce("Paper_ID", "filled_paper_id").alias("publication_id")
)

print("Filled blank Paper_ID")
show(
    studies.filter(pl.col("Paper_ID").is_null()).select(
        "ev_id", "publication_id", "title"
    )
)

if studies.filter(pl.col("publication_id").is_null()).height:
    raise ValueError("A study row has no publication id")

Filled blank Paper_ID
shape: (1, 3)
┌───────────┬────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────┐
│ ev_id     ┆ publication_id ┆ title                                                                                            │
│ ---       ┆ ---            ┆ ---                                                                                              │
│ str       ┆ i64            ┆ str                                                                                              │
╞═══════════╪════════════════╪══════════════════════════════════════════════════════════════════════════════════════════════════╡
│ 62.24.02a ┆ 62             ┆ Risk Sources and Risk Management Measures in Support of Standards for General-Purpose AI Systems │
└───────────┴────────────────┴──────────────────────────────────────────────────────────────────────────────────────────────────┘


In [6]:
mentions = (
    risks.with_columns(
        pl.col("Risk").str.strip_chars().alias("label"),
        pl.col("Assotiated Ev_ID").str.strip_chars().alias("ev_id"),
    )
    .join(studies.select("ev_id", "publication_id"), on="ev_id", how="left")
    .sort("_order")
    .select(pl.col("RISK_ID").alias("id"), "label", "ev_id", "publication_id")
)

if mentions.filter(pl.col("publication_id").is_null()).height:
    raise ValueError("A risk row has no publication id")

mentions.write_csv(MENTIONS)
print(f"{mentions.height} mentions -> {MENTIONS}")

351 mentions -> risk/mentions.csv


In [7]:
ev_titles = studies.sort("_order").select("ev_id", "publication_id", "title")

ev_titles.write_csv(EV_TITLES)
print(f"{ev_titles.height} evidence ids -> {EV_TITLES}")

146 evidence ids -> risk/ev_titles.csv
